# Production d'algues

**Les sources de données viennent de la FAO (format CSV)** : https://www.fao.org/fishery/en/collection/global_production?lang=en

In [1]:
import pandas as pd

In [ ]:
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots

In [ ]:
df_prod = pd.read_csv('../data/Global_production_quantity.csv')
df_species = pd.read_csv('../data/CL_FI_SPECIES_GROUPS.csv')
df_prod.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1152461 entries, 0 to 1152460
Data columns (total 8 columns):
 #   Column                      Non-Null Count    Dtype  
---  ------                      --------------    -----  
 0   COUNTRY.UN_CODE             1152461 non-null  int64  
 1   SPECIES.ALPHA_3_CODE        1152461 non-null  object 
 2   AREA.CODE                   1152461 non-null  int64  
 3   PRODUCTION_SOURCE_DET.CODE  1152461 non-null  object 
 4   MEASURE                     1152461 non-null  object 
 5   PERIOD                      1152461 non-null  int64  
 6   VALUE                       1152461 non-null  float64
 7   STATUS                      1152461 non-null  object 
dtypes: float64(1), int64(3), object(4)
memory usage: 70.3+ MB


**On ne peut identifier la production qu'uniquement à travers le "species.alpha_3_code"**

In [ ]:
df_species.head(10)

,3A_Code,Taxonomic_Code,Identifier,Name_En,Name_Fr,Name_Es,Name_Ar,Name_Cn,Name_Ru,Scientific_Name,...,CPC_Class_Es,CPC_Class_Ar,CPC_Class_Cn,CPC_Class_Ru,CPC_Group_En,CPC_Group_Fr,CPC_Group_Es,CPC_Group_Ar,CPC_Group_Cn,CPC_Group_Ru
0,LAS,1020010XXXXX,2000,Lampreys nei,Lamproies nca,Lampreas nep,NaN,NaN,Миноговые,Petromyzontidae,...,NaN,NaN,NaN,NaN,"Fish live, fresh or chilled for human consumption",NaN,NaN,NaN,NaN,NaN
1,LAR,102001000201,2002,River lamprey,Lamproie de rivière,Lamprea de río,NaN,NaN,Минога речная,Lampetra fluviatilis,...,NaN,NaN,NaN,NaN,"Fish live, fresh or chilled for human consumption",NaN,NaN,NaN,NaN,NaN
2,SBL,105002000201,2003,Bluntnose sixgill shark,Requin griset,Cañabota gris,NaN,NaN,NaN,Hexanchus griseus,...,NaN,NaN,NaN,NaN,"Fish live, fresh or chilled for human consumption",NaN,NaN,NaN,NaN,NaN
3,NTC,105002000301,2004,Broadnose sevengill shark,Platnez,Cañabota gata,NaN,NaN,NaN,Notorynchus cepedianus,...,NaN,NaN,NaN,NaN,"Fish live, fresh or chilled for human consumption",NaN,NaN,NaN,NaN,NaN
4,BSK,106007000101,2005,Basking shark,Pèlerin,Peregrino,NaN,NaN,Акула гигантская,Cetorhinus maximus,...,NaN,NaN,NaN,NaN,"Fish live, fresh or chilled for human consumption",NaN,NaN,NaN,NaN,NaN
5,CCT,106002000101,2006,Sand tiger shark,Requin taureau,Toro bacota,قِرش ثَور,锥齿鲨,NaN,Carcharias taurus,...,NaN,NaN,NaN,NaN,"Fish live, fresh or chilled for human consumption",NaN,NaN,NaN,NaN,NaN
6,THR,1060060001XX,2007,Thresher sharks nei,Renards de mer nca,Zorros nep,NaN,NaN,NaN,Alopias spp,...,NaN,NaN,NaN,NaN,"Fish live, fresh or chilled for human consumption",NaN,NaN,NaN,NaN,NaN
7,ALV,106006000101,2008,Thresher,Renard,Zorro,القرش الثّعلب,狐形长尾鲨,Лисица морская обыкновенная,Alopias vulpinus,...,NaN,NaN,NaN,NaN,"Fish live, fresh or chilled for human consumption",NaN,NaN,NaN,NaN,NaN
8,PTH,106006000102,2009,Pelagic thresher,Renard pélagique,Zorro pelágico,NaN,NaN,NaN,Alopias pelagicus,...,NaN,NaN,NaN,NaN,"Fish live, fresh or chilled for human consumption",NaN,NaN,NaN,NaN,NaN
9,MAK,1060080002XX,2010,Mako sharks,Taupes,Marrajos,السّنفقيات القرشيات,鲭鲨属未定种,NaN,Isurus spp,...,NaN,NaN,NaN,NaN,"Fish live, fresh or chilled for human consumption",NaN,NaN,NaN,NaN,NaN


**Après recherche, c'est dans la colonne "ISSCAAP_group_Fr" qu'on peut repérer les algues et notamment les macro-algues (vertes, brunes, rouges)**

In [ ]:
df_species["ISSCAAP_Group_Fr"].value_counts()

,count
ISSCAAP_Group_Fr,
Poissons côtiers divers,3184
Poissons d'eau douce divers,1422
Poissons démersaux divers,1347
"Squales, raies, chimères",872
"Ormeaux, bigorneaux, strombes",649
"Carpes, barbeaux et autres cyprinidés",616
"Clams, coques, arches",531
Poissons pélagiques divers,425
Crevettes,398


In [ ]:
mask_columns = ["CPC_Class_Es","CPC_Class_Ar","CPC_Class_Cn","CPC_Class_Ru", "ISSCAAP_Group_Cn", "ISSCAAP_Group_Ru", "CPC_Class_Fr","CPC_Group_Fr","CPC_Group_Es", "CPC_Group_Ar","CPC_Group_Cn","CPC_Group_Ru", "ISSCAAP_Group_Es","ISSCAAP_Group_Ar", "Yearbook_Group_Es", "Yearbook_Group_Ar", "Yearbook_Group_Cn", "Yearbook_Group_Ru", "ISSCAAP_Group_En"]
df_species_filter_alguae = df_species.drop(columns=mask_columns)[df_species["ISSCAAP_Group_Fr"].isin(["Algues rouges", "Algues brunes", "Algues vertes"])]
df_species_filter_alguae.head(5)

,3A_Code,Taxonomic_Code,Identifier,Name_En,Name_Fr,Name_Es,Name_Ar,Name_Cn,Name_Ru,Scientific_Name,Author,Major_Group,Yearbook_Group_En,Yearbook_Group_Fr,ISSCAAP_Group_Fr,CPC_Class_En,CPC_Group_En
759,CAU,7410050001XX,2774,Caulerpa seaweeds,Algues caulerpes,Algas caulerpa,ألغيات كوليربا,蕨藻属未定种,NaN,Caulerpa spp,NaN,PLANTAE AQUATICAE,Aquatic plants,Plantes aquatiques,Algues vertes,"Seaweeds and other algae, fresh, frozen or dri...",Other aquatic plants and animals
760,UVP,741008000204,2775,Lacy sea lettuce,NaN,NaN,NaN,NaN,NaN,Ulva australis,Kjellman 1897,PLANTAE AQUATICAE,Aquatic plants,Plantes aquatiques,Algues vertes,"Seaweeds and other algae, fresh, frozen or dri...",Other aquatic plants and animals
761,LNJ,771002000304,2776,Japanese kelp,Laminaire du Japon,Laminaria del Japón,لمنارية اليابان,海带,NaN,Saccharina japonica,"(Areschoug) C.E.Lane, C.Mayes, Druehl & W.Saun...",PLANTAE AQUATICAE,Aquatic plants,Plantes aquatiques,Algues brunes,"Seaweeds and other algae, fresh, frozen or dri...",Other aquatic plants and animals
762,UDP,771004000301,2777,Wakame,Wakamé,Abeto marino,ألغ تنّوبيّ,裙带菜,NaN,Undaria pinnatifida,(Harvey) Suringar 1873,PLANTAE AQUATICAE,Aquatic plants,Plantes aquatiques,Algues brunes,"Seaweeds and other algae, fresh, frozen or dri...",Other aquatic plants and animals
763,EOZ,7710050001XX,2778,NaN,NaN,Chascón nep,NaN,NaN,NaN,Lessonia spp,NaN,PLANTAE AQUATICAE,Aquatic plants,Plantes aquatiques,Algues brunes,"Seaweeds and other algae, fresh, frozen or dri...",Other aquatic plants and animals
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
13396,YVR,787037000101,21360,NaN,NaN,NaN,NaN,NaN,NaN,Gastroclonium ovatum,(Hudson) Papenfuss 1944,PLANTAE AQUATICAE,Aquatic plants,Plantes aquatiques,Algues rouges,"Seaweeds and other algae, fresh, frozen or dri...",Other aquatic plants and animals
13397,FQZ,787039000101,21361,NaN,NaN,NaN,NaN,NaN,NaN,Sarcodia dentata,(Suhr) R.E.Norris ex M.J.Wynne 1989,PLANTAE AQUATICAE,Aquatic plants,Plantes aquatiques,Algues rouges,"Seaweeds and other algae, fresh, frozen or dri...",Other aquatic plants and animals
13607,ZVM,741005000109,21574,NaN,NaN,NaN,NaN,NaN,NaN,Caulerpa nummularia,Harvey ex J.Agardh 1873,PLANTAE AQUATICAE,Aquatic plants,Plantes aquatiques,Algues vertes,"Seaweeds and other algae, fresh, frozen or dri...",Other aquatic plants and animals
13608,UVV,741008000211,21575,NaN,NaN,NaN,NaN,NaN,NaN,Ulva pseudorotundata,"M.Cormaci, G.Furnari & G.Alongi 2014",PLANTAE AQUATICAE,Aquatic plants,Plantes aquatiques,Algues vertes,"Seaweeds and other algae, fresh, frozen or dri...",Other aquatic plants and animals


In [5]:
list_algae_species = df_species_filter_alguae["3A_Code"].tolist()

On a plus qu'à filter le dataframe de production avec cette liste de "3A_code"

In [6]:
df_prod_alguae = df_prod[df_prod["SPECIES.ALPHA_3_CODE"].isin(list_algae_species)]
df_prod_alguae.info()

<class 'pandas.core.frame.DataFrame'>
Index: 9182 entries, 22073 to 1152263
Data columns (total 8 columns):
 #   Column                      Non-Null Count  Dtype  
---  ------                      --------------  -----  
 0   COUNTRY.UN_CODE             9182 non-null   int64  
 1   SPECIES.ALPHA_3_CODE        9182 non-null   object 
 2   AREA.CODE                   9182 non-null   int64  
 3   PRODUCTION_SOURCE_DET.CODE  9182 non-null   object 
 4   MEASURE                     9182 non-null   object 
 5   PERIOD                      9182 non-null   int64  
 6   VALUE                       9182 non-null   float64
 7   STATUS                      9182 non-null   object 
dtypes: float64(1), int64(3), object(4)
memory usage: 645.6+ KB


## Traitement des données de pays

In [7]:
df_countries = pd.read_csv('/content/sample_data/CL_FI_COUNTRY_GROUPS.csv')
df_countries.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 275 entries, 0 to 274
Data columns (total 34 columns):
 #   Column              Non-Null Count  Dtype 
---  ------              --------------  ----- 
 0   UN_Code             275 non-null    int64 
 1   Identifier          275 non-null    int64 
 2   ISO2_Code           257 non-null    object
 3   ISO3_Code           260 non-null    object
 4   Name_En             275 non-null    object
 5   Name_Fr             275 non-null    object
 6   Name_Es             275 non-null    object
 7   Name_Ar             264 non-null    object
 8   Name_Cn             264 non-null    object
 9   Name_Ru             264 non-null    object
 10  Official_Name_En    269 non-null    object
 11  Official_Name_Fr    269 non-null    object
 12  Official_Name_Es    269 non-null    object
 13  Official_Name_Ar    264 non-null    object
 14  Official_Name_Cn    264 non-null    object
 15  Official_Name_Ru    264 non-null    object
 16  Continent_Group_En  263 no

**On décide de "virer" une bonne partie des colonnes notamment celles qui ont des valeurs "null".**

In [ ]:
df_countries.columns

Index(['UN_Code', 'Identifier', 'ISO2_Code', 'ISO3_Code', 'Name_En', 'Name_Fr',
       'Name_Es', 'Name_Ar', 'Name_Cn', 'Name_Ru', 'Official_Name_En',
       'Official_Name_Fr', 'Official_Name_Es', 'Official_Name_Ar',
       'Official_Name_Cn', 'Official_Name_Ru', 'Continent_Group_En',
       'Continent_Group_Fr', 'Continent_Group_Es', 'Continent_Group_Ar',
       'Continent_Group_Cn', 'Continent_Group_Ru', 'EcoClass_Group_En',
       'EcoClass_Group_Fr', 'EcoClass_Group_Es', 'EcoClass_Group_Ar',
       'EcoClass_Group_Cn', 'EcoClass_Group_Ru', 'GeoRegion_Group_En',
       'GeoRegion_Group_Fr', 'GeoRegion_Group_Es', 'GeoRegion_Group_Ar',
       'GeoRegion_Group_Cn', 'GeoRegion_Group_Ru'],
      dtype='object')

In [8]:
df_countries_filter = df_countries[["UN_Code", 'Name_Fr', "Continent_Group_Fr"]]
df_countries_filter

,UN_Code,Name_Fr,Continent_Group_Fr
0,51,Arménie,Asie
1,4,Afghanistan,Asie
2,8,Albanie,Europe
3,12,Algérie,Afrique
4,16,Samoa américaines,Océanie
...,...,...,...
270,728,Soudan du Sud,Afrique
271,680,Sercq,Europe
272,729,Soudan,Afrique
273,275,Palestine,Asie


In [ ]:
def replace_country_name(string):
  if (string == "République de Corée"):
    return "Corée du Sud"
  else:
    return string

In [ ]:
df_countries_filter["Name_Fr"] = df_countries_filter["Name_Fr"].apply(lambda x: replace_country_name(x))

In [ ]:
df_asia = df_countries_filter[df_countries_filter["Continent_Group_Fr"] == "Asie"]
list_asia = df_asia["UN_Code"].tolist()

In [9]:
df_europe = df_countries_filter[df_countries_filter["Continent_Group_Fr"] == "Europe"]
list_europe = df_europe["UN_Code"].tolist()

## Création des graphes algboost

On renomme les types de production pour que ce soit plus simple (juste deux catégories : Récolte et Culture) et quelques autres colonnes.

In [10]:
df_prod_alguae = df_prod_alguae.rename(columns={"PRODUCTION_SOURCE_DET.CODE": "source_production","COUNTRY.UN_CODE": "UN_Code", "PERIOD": "Année","VALUE" : "Production"})
df_prod_alguae["source_production"] = df_prod_alguae["source_production"].apply(lambda x: "Récolte" if x == "CAPTURE" else "Culture")
df_prod_alguae.head(5)

,UN_Code,SPECIES.ALPHA_3_CODE,AREA.CODE,source_production,MEASURE,Année,Production,STATUS
22073,32,SWB,41,Récolte,Q_tlw,2022,0.0,A
22074,32,SWB,41,Récolte,Q_tlw,2021,0.0,A
22075,32,SWB,41,Récolte,Q_tlw,2020,0.0,A
22076,32,SWB,41,Récolte,Q_tlw,2019,0.0,A
22077,32,SWB,41,Récolte,Q_tlw,2018,0.0,A


on fusionne avec le dataframe présentant les pays

In [15]:
df_prod_alguae_country = df_prod_alguae.merge(df_countries_filter, on="UN_Code", how="left")
df_prod_alguae_country.head(5)

,UN_Code,SPECIES.ALPHA_3_CODE,AREA.CODE,source_production,MEASURE,Année,Production,STATUS,Name_Fr,Continent_Group_Fr
0,32,SWB,41,Récolte,Q_tlw,2022,0.0,A,Argentine,Amériques
1,32,SWB,41,Récolte,Q_tlw,2021,0.0,A,Argentine,Amériques
2,32,SWB,41,Récolte,Q_tlw,2020,0.0,A,Argentine,Amériques
3,32,SWB,41,Récolte,Q_tlw,2019,0.0,A,Argentine,Amériques
4,32,SWB,41,Récolte,Q_tlw,2018,0.0,A,Argentine,Amériques


Liste couleurs dispos : ['#1D3B6E','#5B8FCB', '#FDF2ED', '#209490', '#C73175', '#F6A01E', '#1E52A1', '#97C7EC', '#13716A', '#991358', '#E5800B', '#3573B9', '#CDE8FA', '#58B7B0', '#D85F9F', '#F9C06B', '#97C7EC', '#E1D3E9', '#F18882', '#FBDDD9', '#C3ADD4', 'F8C2BB']

 '#0074E4' : FR
 '#1D3B6E' : Chine
 '#209490' : Indonésie
 '#C73175' : Norvège
 '#5B8FCB' : Corée du Sud
 '#C3ADD4' : Philippines
 '#F6A01E' : Autres pays d'Asie
 '#1E52A1' : Autres pays hors Asie
 '#13716A' : Chili
 '#991358' : Inde
 '#E5800B' : Japon
 '#CDE8FA' : Etats-Unis
 '#3573B9' : Autres pays (du monde)
 '#58B7B0' : Irlande
 '#D85F9F' : Islande
 '#F9C06B' : Russie
 '#E1D3E9' : Royaume-Uni
 '#F18882' : Autres pays d'Europe

#### Graphe evolution_production_algues_1950-2022

On doit renommer certains pays et certaines catégories

In [17]:
def get_category(row):
    if row["Continent"] == "Asie":
        if row["Pays_2"] in ["Indonésie", "Philippines", "Corée du Sud", "Chine"]:
            return row["Pays_2"]
        else:
            return "Autres pays d'Asie"
    else:
      return "Autres pays hors Asie"

In [18]:
df_prod_alguae_country_fig_1 = df_prod_alguae_country.rename(columns={"Name_Fr": "Pays_2", "Continent_Group_Fr":"Continent", "Year": "Année"})
df_prod_alguae_country_fig_1.head(5)

,UN_Code,SPECIES.ALPHA_3_CODE,AREA.CODE,source_production,MEASURE,Année,Production,STATUS,Pays_2,Continent
0,32,SWB,41,Récolte,Q_tlw,2022,0.0,A,Argentine,Amériques
1,32,SWB,41,Récolte,Q_tlw,2021,0.0,A,Argentine,Amériques
2,32,SWB,41,Récolte,Q_tlw,2020,0.0,A,Argentine,Amériques
3,32,SWB,41,Récolte,Q_tlw,2019,0.0,A,Argentine,Amériques
4,32,SWB,41,Récolte,Q_tlw,2018,0.0,A,Argentine,Amériques


In [19]:
df_prod_alguae_country_fig_1['Pays'] = df_prod_alguae_country_fig_1.apply(lambda row: get_category(row), axis=1)

In [ ]:
# Il s'agit d'une fconction spécifique pour créer les graphes de type "area chart"
def create_multi_traces_area_plot(df,name_color_tuple_list, plot_title):
  fig = go.Figure()
  for name_color_tuple in name_color_tuple_list:
    fig.add_trace(go.Scatter(
      x=df[df["Pays"]== name_color_tuple[0]]["Année"],
      y=df[df["Pays"]== name_color_tuple[0]]["Production"],
      mode='lines',
      line=dict(width=0.5, color=name_color_tuple[1]),
      stackgroup='one',
      fillcolor=name_color_tuple[1],
      name=name_color_tuple[0],
      hovertemplate="""Production : %{y}""" + \
      """<extra></extra>""",
      legendwidth=500,
      hoverlabel=dict(font=dict(family="Parkinsans")),
      )
    )
  fig.update_layout(
    legend_orientation="h",
    hovermode='x unified',
    title_text=plot_title,
    font_family="Parkinsans",
    title_font_weight=800,
    font_weight=600,
    font_color="#1D3B6E",
    paper_bgcolor = 'white',  # Fully transparent background
    plot_bgcolor = 'white',   # Fully transparent plot area
  )
  return fig

In [26]:
df_prod_alguae_country_fig_1_by_year = df_prod_alguae_country_fig_1[["Année", "Pays","Production"]].groupby(["Année", "Pays"]).sum().reset_index()

colors = [('Chine','#1D3B6E'),('Indonésie','#209490'),('Corée du Sud','#5B8FCB'),('Philippines','#C3ADD4'), ("Autres pays d'Asie", '#F6A01E'), ("Autres pays hors Asie",'#1E52A1')]

fig_1 = create_multi_traces_area_plot(df_prod_alguae_country_fig_1_by_year,colors, "Evolution de la production d'algues<br>(en tonnes) par pays")
fig_1

In [27]:
fig_1.write_html("evolution_production_algues_1950-2022_mobile.html", include_plotlyjs="cdn")

#### Graphe "camembert" repartition_culture_recolte Monde et Europe

In [28]:
df_prod_alguae_country_fig_3 = df_prod_alguae_country.rename(columns={"Name_Fr": "Pays", "Continent_Group_Fr":"Continent"})

In [32]:
# import math
# colors= {'Culture':'#209490',
#      'Récolte':'#5B8FCB'}
#209490 (vert) #5B8FCB' (bleu)
colors= ['#209490','#5B8FCB' ]
df_prod_alguae_country_fig_3_europe = df_prod_alguae_country_fig_3[df_prod_alguae_country_fig_3["Continent"] == "Europe"]
data_go = df_prod_alguae_country_fig_3[["source_production", "Production"]].groupby("source_production").sum().reset_index()

fig_3 = go.Figure(data=[go.Pie(labels=data_go["source_production"],
                             values=data_go["Production"],
                               pull=[0.2,0],
                               texttemplate = "%{label} <br> %{percent:.0%}",
                               )])
fig_3.update_traces(hoverinfo='skip',textfont_size=14,
                  marker=dict(colors=colors), textposition='outside')
fig_3.update_layout(
    showlegend=False,
    title_text="Répartition entre récolte et<br>culture d'algues dans le monde",
    font_family="Parkinsans",
    title_font_weight=800,
    font_weight=600,
    font_color="#1D3B6E",
)

data_go_bis = df_prod_alguae_country_fig_3_europe[["source_production", "Production"]].groupby("source_production").sum().reset_index()

fig_3_bis = go.Figure(data=[go.Pie(labels=data_go_bis["source_production"],
                             values=data_go_bis["Production"],
                                   pull=[0.2, 0],
                                    texttemplate = "%{label} <br> %{percent:.0%}"
                               )])
fig_3_bis.update_traces(hoverinfo='skip', textfont_size=14,
                  marker=dict(colors=colors), textposition='outside')
fig_3_bis.update_layout(
    font_family="Parkinsans",
    title_font_weight=800,
    font_weight=600,
    font_color="#1D3B6E",
    showlegend=False,
    title_text="Répartition entre récolte et<br>culture d'algues en Europe"
)
display(fig_3,fig_3_bis)
# fig_3
# fig_3 = go.Figure(data=[go.Pie(labels=["Hello", "Bonjour", "Salut"],
#                              values=[1,2,4],pull=[0.2,0]
#                                )])
# fig_3.update_traces(hoverinfo='skip', textinfo='label+percent', textfont_size=14,
#                   marker=dict(colors=colors), textposition='outside')

In [33]:
fig_3.write_html("repartition_culture_recolte_monde_mobile.html", include_plotlyjs="cdn")
fig_3_bis.write_html("repartition_culture_recolte_europe_mobile.html", include_plotlyjs="cdn")

#### Graphe recolte_algues_monde_par_pays

In [34]:
df_prod_alguae_country_fig_4 = df_prod_alguae_country.rename(columns={"Name_Fr": "Pays_2", "Continent_Group_Fr":"Continent"})
df_prod_alguae_country_fig_4_recolte = df_prod_alguae_country_fig_4[df_prod_alguae_country_fig_4["source_production"] == "Récolte"]

In [35]:
def get_category_bis(row):
    if row["Pays_2"] in ["Chili", "Norvège", "Japon", "Inde", "Indonésie", "France"]:
       return row["Pays_2"]
    elif row["Pays_2"].lower() == "états-unis d'amérique":
      return "Etats-Unis"
    else:
      return "Autres pays"

Liste couleurs dispos : ['#1D3B6E','#5B8FCB', '#FDF2ED', '#209490', '#C73175', '#F6A01E', '#1E52A1', '#97C7EC', '#13716A', '#991358', '#E5800B', '#3573B9', '#CDE8FA', '#58B7B0', '#D85F9F', '#F9C06B', '#97C7EC', '#E1D3E9', '#F18882', '#FBDDD9', '#C3ADD4', 'F8C2BB']

 '#0074E4' : FR
 '#1D3B6E' : Chine
 '#209490' : Indonésie
 '#C73175' : Norvège
 '#5B8FCB' : Corée du Sud
 '#97C7EC' : Philippines
 '#F6A01E' : Autres pays d'Asie
 '#1E52A1' : Autres pays hors Asie
 '#13716A' : Chili
 '#991358' : Inde
 '#E5800B' : Japon
 '#CDE8FA' : Etats-Unis
 '#3573B9' : Autres pays (du monde)
 '#58B7B0' : Irlande
 '#D85F9F' : Islande
 '#F9C06B' : Russie
 '#E1D3E9' : Royaume-Uni
 '#F18882' : Autres pays d'Europe

In [40]:
df_prod_alguae_country_fig_4_recolte['Pays'] = df_prod_alguae_country_fig_4_recolte.apply(lambda row: get_category_bis(row), axis=1)
df_prod_alguae_country_fig_4_recolte_by_year = df_prod_alguae_country_fig_4_recolte[["Année", "Pays","Production"]].groupby(["Année", "Pays"]).sum().reset_index()
colors = [('Chili','#13716A'),('Norvège','#C73175'),("Indonésie",'#209490'),('Inde','#991358'), ('Japon', '#E5800B'), ('France','#0074E4'), ("Etats-Unis", '#CDE8FA'), ('Autres pays', '#3573B9')]

fig_4 = create_multi_traces_area_plot(df_prod_alguae_country_fig_4_recolte_by_year,colors, "Récolte d'algues dans le monde<br>(en tonnes) par pays")
fig_4

/tmp/ipykernel_16607/2723511226.py:1: SettingWithCopyWarning:


A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy



In [41]:
fig_4.write_html("recolte_algues_monde_par_pays_mobile.html", include_plotlyjs="cdn")

#### Graphe production_algues_europe_par_pays

In [42]:
def get_country_europe(row):
    if row["Pays_2"] in ["Norvège", "France", "Irlande", "Islande"]:
       return row["Pays_2"]
    elif row["Pays_2"] == "Fédération de Russie":
      return "Russie"
    elif row["Pays_2"] == "Royaume-Uni de Grande-Bretagne et d'Irlande du Nord":
      return "Royaume-Uni"
    else:
      return "Autres pays d'Europe"

In [ ]:
df_prod_alguae_europe = df_prod_alguae[df_prod_alguae["UN_Code"].isin(list_europe)].rename(columns={"COUNTRY.UN_CODE": "UN_Code", "PERIOD": "Année","VALUE" : "Production"}).merge(df_countries_filter, on="UN_Code", how="left").rename(columns={"Name_Fr": "Pays_2", "Continent_Group_Fr": "Continent"})
df_prod_alguae_europe["Pays"] = df_prod_alguae_europe.apply(lambda row: get_country_europe(row), axis=1)
df_prod_alguae_europe_country_by_year = df_prod_alguae_europe[["Année", "Pays", "Production"]].groupby(["Année", "Pays"]).sum().reset_index()

colors = [('Norvège','#C73175'),('France','#0074E4'),('Irlande','#58B7B0'),('Islande','#D85F9F'), ("Russie",'#F9C06B'), ('Royaume-Uni','#E1D3E9'), ("Autres pays d'Europe",'#F18882')]

fig_5 = create_multi_traces_area_plot(df_prod_alguae_europe_country_by_year,colors, plot_title="Production d'algues en Europe<br>(en tonnes) par pays")
fig_5

In [48]:
fig_5.write_html("production_algues_europe_par_pays_mobile.html", include_plotlyjs="cdn")

#### Graphe 6 algoboost (répartition de la prodcution d'espèces d'algues selon Récolte/Aquaculture)

In [ ]:
list_algae_culture = ['Alaria sp.','Ascophyllum nodosum','Asparagopsis sp.','Brown seaweed','Calliblepharis jubata','Caulerpa sp.','Chondrus sp.','Codium sp.','Falkenbergia sp.','Fucus sp.','Gracilaria sp.','Gracilariopsis longissima','Himanthalia sp.','Kappaphycus','Laminaria sp.','Palmaria sp.','Porphyra sp.','Saccharina sp.','Schizymenia jonssonii','Ulva sp.','Ulvella lens','Undaria sp.','Espèce inconnue','Vertebrata lanosa']
list_nb_culture = [17,1,1,2,1,1,2,2,1,5,3,2,2,1,10,8,2,29,1,15,1,3,3,1]

In [54]:
df_culture = pd.DataFrame({"algae":list_algae_culture, "nb_culture": list_nb_culture })
df_culture_sort = df_culture.sort_values(by="nb_culture", ascending=False)
df_culture_sort

,algae,nb_culture
17,Saccharina sp.,29
0,Alaria sp.,17
19,Ulva sp.,15
14,Laminaria sp.,10
15,Palmaria sp.,8
9,Fucus sp.,5
22,Espèce inconnue,3
21,Undaria sp.,3
10,Gracilaria sp.,3
16,Porphyra sp.,2


In [55]:
list_algae_recolte = ['Alaria sp.','Ascophyllum nodosum','Asparagopsis sp.','Bifurcaria bifurcata','Brown seaweed','Calcareous algae','Chondrus sp.','Chorda filum','Coccotylus truncatus','Codium sp.','Corallina sp.','Cystoseira sp.','Delesseria sanguinea','Dilsea carnosa','Fucus sp.','Furcellaria lumbricalis','Gelidium sp.','Gigartina sp.','Gracilaria sp.','Grateloupia turuturu','Green seaweed','Halopteris scoparia','Himanthalia sp.','Laminaria sp.','Lithothamnium calcareum','Mastocarpus stellatus','Osmundea pinnatifida','Padina pavonica','Palmaria sp.','Pelvetia sp.','Petalonia binghamiae','Porphyra sp.','Pterocladiella capillacea','Red seaweed','Saccharina sp.','Salicornia sp.','Sargassum sp.','Solieria sp.','Ulva sp.','Undaria sp.','Espèce inconnue','Vertebrata lanosa','Zonaria tournefortii']
list_nb_recolte =[13,25,2,1,7,1,21,1,1,6,1,1,1,1,30,3,3,3,1,1,2,1,30,37,4,3,5,1,36,1,1,25,1,3,26,2,2,1,32,22,10,4,1]

In [56]:
df_autre_recolte = pd.DataFrame({"algae":list_algae_recolte, "nb_recolte": list_nb_recolte })
df_autre_recolte

,algae,nb_recolte
0,Alaria sp.,13
1,Ascophyllum nodosum,25
2,Asparagopsis sp.,2
3,Bifurcaria bifurcata,1
4,Brown seaweed,7
5,Calcareous algae,1
6,Chondrus sp.,21
7,Chorda filum,1
8,Coccotylus truncatus,1
9,Codium sp.,6


In [59]:
df_autre_recolte_sort = df_autre_recolte.sort_values(by="nb_recolte", ascending=False)
df_autre_recolte_sort

,algae,nb_recolte
23,Laminaria sp.,37
28,Palmaria sp.,36
38,Ulva sp.,32
14,Fucus sp.,30
22,Himanthalia sp.,30
34,Saccharina sp.,26
31,Porphyra sp.,25
1,Ascophyllum nodosum,25
39,Undaria sp.,22
6,Chondrus sp.,21


In [60]:
colors_recolte = ['#1D3B6E','#5B8FCB', '#FDF2ED', '#209490', '#C73175', '#F6A01E', '#1E52A1', '#97C7EC', '#13716A', '#991358', '#E5800B', '#3573B9', '#CDE8FA', '#58B7B0', '#D85F9F', '#F9C06B', '#97C7EC', '#E1D3E9', '#F18882', '#FBDDD9', '#C3ADD4', 'F8C2BB']
colors_culture = ['#F6A01E','#E5800B', '#FDF2ED', '#1D3B6E', '#5B8FCB','#209490' , '#3573B9', '#13716A','#F9C06B', '#1E52A1','#D85F9F', '#C73175', '#58B7B0', '#991358', '#CDE8FA','#97C7EC'  , '#97C7EC', '#E1D3E9', '#F18882', '#FBDDD9', '#C3ADD4', 'F8C2BB']

In [68]:

## Récolte
fig_6 = go.Figure()
fig_6.add_trace(go.Treemap(
    labels = df_autre_recolte_sort["algae"],
    parents = ["",] *15,
    values =  df_autre_recolte_sort["nb_recolte"],
    texttemplate="""<span style="font-weight:800">%{label}</span> <br>""" + \
    """%{value} entreprises (%{percentEntry:.1%})""",
    marker_colors= colors_recolte,
    root_color="white"

))
fig_6.update_layout(
    title_text="Espèces d'algues produites<br>en Europe (Récolte)",
    font_family="Parkinsans",
    title_font_weight=800,
    font_weight=600,
    font_color="#1D3B6E",
  )
fig_6.update_traces(marker=dict(cornerradius=5), hoverinfo="skip")

## Culture
fig_6_bis = go.Figure()
fig_6_bis.add_trace(go.Treemap(
    labels = df_culture_sort["algae"],
    parents = ["",] *15,
    # parents = ["",] *24,
    values = df_culture_sort["nb_culture"],
     texttemplate="""<span style="font-weight:800">%{label}</span> <br>""" + \
    """%{value} entreprises (%{percentEntry:.1%})""",
    marker_colors= colors_culture,
    root_color="white",

))
fig_6_bis.update_layout(
    title_text="Espèces d'algues produites<br>en Europe (Culture)",
    font_family="Parkinsans",
    title_font_weight=800,
    font_weight=600,
    font_color="#1D3B6E",
  )

fig_6_bis.update_traces(marker=dict(cornerradius=5), hoverinfo="skip")
display(
    fig_6,
    fig_6_bis
)

In [69]:
fig_6.write_html("especes_algues_produites_europe_recolte_mobile.html", include_plotlyjs="cdn")

In [70]:
fig_6_bis.write_html("especes_algues_produites_europe_culture_mobile.html", include_plotlyjs="cdn")

#### Graphe 7 algoboost

In [76]:

x=['Emplois<br>directs', 'Emplois<br>indirects', 'Emplois<br>induits', 'Total<br>Emplois']
df =pd.DataFrame({"x":x, "y":[0, 37, 65, 0], "z":[22,17,11,50], "a": [15, 11, 7, 33] })
# df

fig_7 = go.Figure(go.Bar(x=df["x"], y=df["y"], name='',marker_color='white')
    )
fig_7.add_trace(go.Bar(x=df["x"], y=df["z"], name='Produits cultivés et finis en Europe', marker_color='#F6A01E', text=df["z"], textfont=dict(color="white",size=10)) )
fig_7.add_trace(go.Bar(x=df["x"], y=df["a"], name='Produits cultivés hors Europe, finis en Europe',marker_color='#209490', text=df["a"], textfont=dict(color="white", size=10)))

fig_7.update_layout(barmode='stack',
                  legend_orientation='h',
                  paper_bgcolor = 'white',  # Fully transparent background
                  plot_bgcolor = 'white',   # Fully transparent plot area
                  # xaxis={'categoryorder':'category ascending'}
                  title_text="Emplois potentiels créés<br>(milliers de ETP) en Europe<br>par l'industrie des algues en 2030",
                  font_family="Parkinsans",
                  title_font_weight=800,
                  font_weight=600,
                  font_color="#1D3B6E",
                  # legend=dict(yanchor="bottom",
                  #     y=0,
                  #     xanchor="center",
                  #     x=0.5)
    )

fig_7.update_yaxes(showline=False, visible=False)
fig_7.update_xaxes(showline=True, linewidth=2, linecolor='#1D3B6E', tickfont=dict(size=10))
fig_7.update_traces(hoverinfo="skip", marker=dict(
                              line=dict(width=1,
                                        color='rgba(0,0,0,0)'),
                              ), textposition='inside'
)
fig_7.show()

In [77]:
fig_7.write_html("projection_emplois_2030_mobile.html", include_plotlyjs="cdn")

### Graphe production algues en Asie

In [ ]:
df_prod_alguae_asia = df_prod_alguae[df_prod_alguae["COUNTRY.UN_CODE"].isin(list_asia)].rename(columns={"COUNTRY.UN_CODE": "UN_Code", "PERIOD": "Year","VALUE" : "Production"}).merge(df_countries_filter, on="UN_Code", how="left").rename(columns={"Name_Fr": "Pays", "Continent_Group_Fr": "Continent"})
df_prod_alguae_asia["Production"] = df_prod_alguae_asia["Production"]/1000000
df_prod_alguae_asia_countryYear = df_prod_alguae_asia[["Year", "Pays", "Production"]].groupby(["Year", "Pays"]).sum().reset_index()
df_prod_alguae_asia_countryYear

,Year,Pays,Production
0,1950,Chine,0.000000
1,1950,Chine - RAS de Hong-Kong,0.000050
2,1950,Indonésie,0.001000
3,1950,Japon,0.145400
4,1950,Philippines,0.000585
...,...,...,...
844,2022,République populaire démocratique de Corée,0.603000
845,2022,Sri Lanka,0.000271
846,2022,Timor-Leste,0.000700
847,2022,Türkiye,0.000020


In [ ]:
fig = px.line(df_prod_alguae_asia_countryYear, x="Year", y="Production", title="Production d'algues (en millions de tonnes) par pays en Asie de 1950 à 2022", markers=True, color="Pays")
fig.show()

### Graphe production algues en Europe

In [ ]:
df_prod_alguae_europe = df_prod_alguae[df_prod_alguae["COUNTRY.UN_CODE"].isin(list_europe)].rename(columns={"COUNTRY.UN_CODE": "UN_Code", "PERIOD": "Year","VALUE" : "Production"}).merge(df_countries_filter, on="UN_Code", how="left").rename(columns={"Name_Fr": "Pays", "Continent_Group_Fr": "Continent"})
df_prod_alguae_europe["Production"] = df_prod_alguae_europe["Production"]/1000
df_prod_alguae_europe_countryYear = df_prod_alguae_europe[["Year", "Pays", "Production"]].groupby(["Year", "Pays"]).sum().reset_index()
df_prod_alguae_europe_countryYear

,Year,Pays,Production
0,1950,Bulgarie,0.000000
1,1950,Espagne,16.000000
2,1950,France,35.000000
3,1950,Islande,10.000000
4,1950,Italie,1.000000
...,...,...,...
762,2022,Islande,18.300000
763,2022,Italie,1.200000
764,2022,Norvège,171.362482
765,2022,Portugal,1.223900


In [ ]:
fig = px.line(df_prod_alguae_europe_countryYear, x="Year", y="Production", title="Production d'algues (en milliers de tonnes) par pays en Europe de 1950 à 2022", markers=True, color="Pays")
fig.show()

### Graphe répartition type production des pays d'Asie (2022)

In [ ]:
df_prod_alguae_asia_typeprod = df_prod_alguae_asia[df_prod_alguae_asia["Year"] == 2022].rename(columns={"PRODUCTION_SOURCE_DET.CODE": "source_production"})

fig = px.histogram(df_prod_alguae_asia_typeprod, x="Pays", y="Production", title="Repartition des sources de production d'algues (en millions de tonnes) en Asie (2022)", color="source_production").update_xaxes(categoryorder='total descending')
fig.show()

### Graphe répartition type production des pays d'Europe (2022)

In [ ]:
df_prod_alguae_europe_typeprod = df_prod_alguae_europe[df_prod_alguae_europe["Year"] == 2022].rename(columns={"PRODUCTION_SOURCE_DET.CODE": "source_production"})
df_prod_alguae_europe_typeprod["source_production"] = df_prod_alguae_europe_typeprod["source_production"].apply(lambda x: "Récolte" if x == "CAPTURE" else "Aquaculture")
fig = px.histogram(df_prod_alguae_europe_typeprod, x="Pays", y="Production", title="Repartition des sources de production d'algues (en tonnes) en Europe (2022)", color="source_production").update_xaxes(categoryorder='total descending')
fig.show()

### Graphe global de répartition type production selon les continents (2022)

In [ ]:
df_prod_alguae_typeProd_global = df_prod_alguae[df_prod_alguae["PERIOD"] == 2022].rename(columns={"PRODUCTION_SOURCE_DET.CODE": "source_production", "COUNTRY.UN_CODE": "UN_Code", "PERIOD": "Year", "VALUE": "Production"}).merge(df_countries_filter, on="UN_Code", how="left").rename(columns={"Name_Fr": "Pays", "Continent_Group_Fr": "Continent"})
df_prod_alguae_typeProd_global

,UN_Code,SPECIES.ALPHA_3_CODE,AREA.CODE,source_production,MEASURE,Year,Production,STATUS,Pays,Continent
0,32,SWB,41,CAPTURE,Q_tlw,2022,0.00,A,Argentine,Amériques
1,32,SWG,41,CAPTURE,Q_tlw,2022,0.00,A,Argentine,Amériques
2,32,APL,41,CAPTURE,Q_tlw,2022,0.00,A,Argentine,Amériques
3,36,SWB,57,CAPTURE,Q_tlw,2022,1923.00,I,Australie,Océanie
4,124,ASN,21,CAPTURE,Q_tlw,2022,12097.00,A,Canada,Amériques
...,...,...,...,...,...,...,...,...,...,...
233,704,EMA,71,MARINE,Q_tlw,2022,707.54,A,Viet Nam,Asie
234,704,GLS,71,MARINE,Q_tlw,2022,8907.30,A,Viet Nam,Asie
235,704,SWG,71,BRACKISHWATER,Q_tlw,2022,900.00,I,Viet Nam,Asie
236,836,EMA,51,MARINE,Q_tlw,2022,1000.00,I,"République-Unie de Tanzanie, Zanzibar",Afrique


In [ ]:

colors= {'Aquaculture':'royalblue',
    'Récolte':'tomato'}

df_prod_alguae_typeProd_global["source_production"] = df_prod_alguae_typeProd_global["source_production"].apply(lambda x: "Récolte" if x == "CAPTURE" else "Aquaculture")
df_prod_alguae_typeProd_asia = df_prod_alguae_typeProd_global[df_prod_alguae_typeProd_global["Continent"] == "Asie"]
df_prod_alguae_typeProd_europe = df_prod_alguae_typeProd_global[df_prod_alguae_typeProd_global["Continent"] == "Europe"]
df_prod_alguae_typeProd_amerique = df_prod_alguae_typeProd_global[df_prod_alguae_typeProd_global["Continent"] == "Amériques"]
df_prod_alguae_typeProd_africa = df_prod_alguae_typeProd_global[df_prod_alguae_typeProd_global["Continent"] == "Afrique"]
df_prod_alguae_typeProd_oceania = df_prod_alguae_typeProd_global[df_prod_alguae_typeProd_global["Continent"] == "Océanie"]
fig = px.pie(df_prod_alguae_typeProd_asia, values='Production', names='source_production', color="source_production", title='Repartition production Asie (2022)', color_discrete_map=colors)
fig_2 = px.pie(df_prod_alguae_typeProd_europe, values='Production', names='source_production',color="source_production", title='Repartition production Europe (2022)', color_discrete_map=colors)
fig_3 = px.pie(df_prod_alguae_typeProd_amerique, values='Production', names='source_production', color="source_production",title='Repartition production Amérique (Nord et Sud) (2022)',color_discrete_map=colors)
fig_4 = px.pie(df_prod_alguae_typeProd_africa, values='Production', names='source_production', color="source_production",title='Repartition production Afrique (2022)',color_discrete_map=colors)
fig_5 = px.pie(df_prod_alguae_typeProd_oceania, values='Production', names='source_production',color="source_production", title='Repartition production Océanie (2022)',color_discrete_map=colors)
display(fig, fig_2, fig_3, fig_4, fig_5)




In [ ]:
prod_asia = df_prod_alguae_typeProd_asia["Production"].sum()
prod_asia

np.float64(36653680.48)

In [ ]:
prod_europe = df_prod_alguae_typeProd_europe["Production"].sum()
prod_europe

np.float64(321244.85099999997)

## Vérification de la production française d'algues

**Dans différents rapports européens ils reprennent plusieurs fois le chiffre d'environ 51 000 tonnes d'algues produites en France en 2019 (alors que je trouve plutôt 34 000 tonnes), pourquoi cette différence ? (alors que je trouve des chiffres identiques pour les autres pays)**

In [ ]:
df_country_france = df_countries_filter[df_countries_filter["Name_Fr"] == "France"]
df_country_france

,UN_Code,Name_Fr,Continent_Group_Fr
67,250,France,Europe


In [ ]:
df_prod_france = df_prod[df_prod["COUNTRY.UN_CODE"] == 250]

In [ ]:
df_prod_alguae_france_2019 = df_prod_france[df_prod_france['SPECIES.ALPHA_3_CODE'].isin(list_algae_species)&(df_prod_france["PERIOD"] == 2019)]
prod_algua_france_2019 = df_prod_alguae_france_2019["VALUE"].sum()
prod_algua_france_2019

np.float64(34331.114)

**J'essaie de refaire ici le calcul de la production d'algues en France en 2019, je trouve la même chose que sur le graphe. Qu'est ce que les aueturs des rapports ont pu rajouter pour avoir cette production ? (surtout que l'on trouve des résulttas identiques pour les autres pays, y'at-il eu en quelques années une grosse maj sur les données FR ? Un problème de classification dans les listes d'espèces qui pénalisent fortement la prodcution française et qui n'est pas prise en compte ?)**

In [ ]:
df_species_filter_alguae

,3A_Code,Taxonomic_Code,Identifier,Name_En,Name_Fr,Name_Es,Name_Ar,Name_Cn,Name_Ru,Scientific_Name,Author,Major_Group,Yearbook_Group_En,Yearbook_Group_Fr,ISSCAAP_Group_Fr,CPC_Class_En,CPC_Group_En
758,SIZ,7110010001XX,2773,Spirulina nei,Spirulina nca,Spirulina nep,NaN,NaN,NaN,Spirulina spp,NaN,PLANTAE AQUATICAE,Aquatic plants,Plantes aquatiques,Plantes aquatiques diverses,"Seaweeds and other algae, fresh, frozen or dri...",Other aquatic plants and animals
759,CAU,7410050001XX,2774,Caulerpa seaweeds,Algues caulerpes,Algas caulerpa,ألغيات كوليربا,蕨藻属未定种,NaN,Caulerpa spp,NaN,PLANTAE AQUATICAE,Aquatic plants,Plantes aquatiques,Algues vertes,"Seaweeds and other algae, fresh, frozen or dri...",Other aquatic plants and animals
760,UVP,741008000204,2775,Lacy sea lettuce,NaN,NaN,NaN,NaN,NaN,Ulva australis,Kjellman 1897,PLANTAE AQUATICAE,Aquatic plants,Plantes aquatiques,Algues vertes,"Seaweeds and other algae, fresh, frozen or dri...",Other aquatic plants and animals
761,LNJ,771002000304,2776,Japanese kelp,Laminaire du Japon,Laminaria del Japón,لمنارية اليابان,海带,NaN,Saccharina japonica,"(Areschoug) C.E.Lane, C.Mayes, Druehl & W.Saun...",PLANTAE AQUATICAE,Aquatic plants,Plantes aquatiques,Algues brunes,"Seaweeds and other algae, fresh, frozen or dri...",Other aquatic plants and animals
762,UDP,771004000301,2777,Wakame,Wakamé,Abeto marino,ألغ تنّوبيّ,裙带菜,NaN,Undaria pinnatifida,(Harvey) Suringar 1873,PLANTAE AQUATICAE,Aquatic plants,Plantes aquatiques,Algues brunes,"Seaweeds and other algae, fresh, frozen or dri...",Other aquatic plants and animals
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
13608,UVV,741008000211,21575,NaN,NaN,NaN,NaN,NaN,NaN,Ulva pseudorotundata,"M.Cormaci, G.Furnari & G.Alongi 2014",PLANTAE AQUATICAE,Aquatic plants,Plantes aquatiques,Algues vertes,"Seaweeds and other algae, fresh, frozen or dri...",Other aquatic plants and animals
13609,KGW,787022001301,21576,Pitcher Siphon Weed,NaN,NaN,NaN,NaN,NaN,Polysiphonia stricta,(Mertens ex Dillwyn) Greville 1824,PLANTAE AQUATICAE,Aquatic plants,Plantes aquatiques,Algues rouges,"Seaweeds and other algae, fresh, frozen or dri...",Other aquatic plants and animals
13610,ZWJ,799001000201,21577,NaN,NaN,NaN,NaN,NaN,NaN,Nannochloropsis oculata,(Droop) D.J.Hibberd 1981,PLANTAE AQUATICAE,Aquatic plants,Plantes aquatiques,Plantes aquatiques diverses,"Seaweeds and other algae, fresh, frozen or dri...",Other aquatic plants and animals
13611,ZVV,794000100101,21578,NaN,NaN,NaN,NaN,NaN,NaN,Diacronema lutheri,(Droop) Bendif & Véron 2011,PLANTAE AQUATICAE,Aquatic plants,Plantes aquatiques,Plantes aquatiques diverses,"Seaweeds and other algae, fresh, frozen or dri...",Other aquatic plants and animals


## Tests

In [ ]:
df_prod_algae_2022 = df_prod_alguae[df_prod_alguae["PERIOD"] == 2022].rename(columns={"COUNTRY.UN_CODE": "UN_Code", "PERIOD": "Année","VALUE" : "Production", "SPECIES.ALPHA_3_CODE": "3A_Code", "PRODUCTION_SOURCE_DET.CODE": "source_production"})
df_prod_algae_2022

,UN_Code,3A_Code,AREA.CODE,source_production,MEASURE,Année,Production,STATUS
22073,32,SWB,41,CAPTURE,Q_tlw,2022,0.00,A
22230,32,SWG,41,CAPTURE,Q_tlw,2022,0.00,A
22289,32,APL,41,CAPTURE,Q_tlw,2022,0.00,A
42454,36,SWB,57,CAPTURE,Q_tlw,2022,1923.00,I
105052,124,ASN,21,CAPTURE,Q_tlw,2022,12097.00,A
...,...,...,...,...,...,...,...,...
1151372,704,EMA,71,MARINE,Q_tlw,2022,707.54,A
1151397,704,GLS,71,MARINE,Q_tlw,2022,8907.30,A
1151470,704,SWG,71,BRACKISHWATER,Q_tlw,2022,900.00,I
1152209,836,EMA,51,MARINE,Q_tlw,2022,1000.00,I


## Top 20 espèces d'algues produites dans le Monde + constitution graphe camembert pour site Algoboost

In [ ]:
df_prod_algae_2022_by_species_top_20 = df_prod_algae_2022[["Production", "3A_Code"]].groupby("3A_Code").sum().sort_values(by="Production", ascending=False).head(20)
df_prod_algae_2022_by_species_top_20 = df_prod_algae_2022_by_species_top_20.merge(df_species_filter_alguae[["3A_Code", "Scientific_Name"]], on="3A_Code", how="left").rename(columns={"Scientific_Name": "Espèce"})
df_prod_algae_2022_by_species_top_20

,3A_Code,Production,Espèce
0,LNJ,1.090225e+07,Saccharina japonica
1,EMX,7.803852e+06,Eucheuma spp
2,GLS,7.608578e+06,Gracilaria spp
3,UDP,2.699952e+06,Undaria pinnatifida
4,FYS,2.176885e+06,Porphyra spp
5,SWB,2.146560e+06,Phaeophyceae
6,EMA,1.804133e+06,Kappaphycus alvarezii
7,PRT,7.860540e+05,Pyropia tenera
8,GQB,3.471635e+05,Sargassum fusiforme
9,EMI,2.353102e+05,Eucheuma denticulatum


In [ ]:
fig = px.histogram(df_prod_algae_2022_by_species_top_20,x="Espèce", y="Production", color="Espèce", title="Repartition de la Production 2022 (en tonnes) dans le monde selon l'espèce d'algues")
fig.show()

Si on prend les 10 premières algues en production on a  :
1. saccharina japonica (Kombu = alimentation)
2. eucheuma spp (gélifiants cosmétiques/agroalimentaire/pharmacie)
3. gracilaria spp (agar additifs agroalimentaire)
4. Undaria pinnatifida (Wakame = alimentation/cosmétiques/pharmacie)
5. Porphyra spp (Nori = alimentation)
6. Phaeophyceae (algues brunes = fertilisants agriculture)
7. Kappaphycus alvarezi (algue rouge = additifs agroalimentaires/pharmacie)
8. Pyropia tenera (nori rouge ? = alimentation)
9. Sargassum fusiforme (hijiki = alimentation)
10. Eucheuma denticulatum (gélifiants cosmétiques/agroalimentaire/pharmacie).

In [ ]:
df_prod_algae_2022_by_species_top_10 = df_prod_algae_2022_by_species_top_20[["Espèce", "Production"]].head(10)
df_prod_algae_2022_by_species_top_10

,Espèce,Production
0,Saccharina japonica,1.090225e+07
1,Eucheuma spp,7.803852e+06
2,Gracilaria spp,7.608578e+06
3,Undaria pinnatifida,2.699952e+06
4,Porphyra spp,2.176885e+06
5,Phaeophyceae,2.146560e+06
6,Kappaphycus alvarezii,1.804133e+06
7,Pyropia tenera,7.860540e+05
8,Sargassum fusiforme,3.471635e+05
9,Eucheuma denticulatum,2.353102e+05


On essaye de "répartir" la production en focntion du type d'applications (si une production a 3 type sd'application on met un tiers dans chaque par exemple)

In [ ]:
prod_alimentation = df_prod_algae_2022_by_species_top_10["Production"][0] + df_prod_algae_2022_by_species_top_10["Production"][3] / 3 + df_prod_algae_2022_by_species_top_10["Production"][4] + df_prod_algae_2022_by_species_top_10["Production"][7] + df_prod_algae_2022_by_species_top_10["Production"][8]
prod_cosmetiques = df_prod_algae_2022_by_species_top_10["Production"][1] / 3 + df_prod_algae_2022_by_species_top_10["Production"][3] / 3 + df_prod_algae_2022_by_species_top_10["Production"][9] / 2
prod_agroalim = df_prod_algae_2022_by_species_top_10["Production"][1] / 3 + df_prod_algae_2022_by_species_top_10["Production"][2] + df_prod_algae_2022_by_species_top_10["Production"][6] / 2 + df_prod_algae_2022_by_species_top_10["Production"][9] / 3
prod_agriculture = df_prod_algae_2022_by_species_top_10["Production"][5]
prod_pharma = df_prod_algae_2022_by_species_top_10["Production"][1] / 3 + df_prod_algae_2022_by_species_top_10["Production"][3] / 3 + df_prod_algae_2022_by_species_top_10["Production"][6] / 2 + df_prod_algae_2022_by_species_top_10["Production"][9] / 3

On recrée un tableau propre avec les nouvelles productions selon l'application

In [ ]:
new_df_applications = pd.DataFrame({"Application": ["Alimentation", "Cosmétiques", "Agroalimentaire", "Agriculture", "Pharmacie"], "Proportion": [prod_alimentation,prod_cosmetiques,prod_agroalim, prod_agriculture, prod_pharma]})
new_df_applications

,Application,Proportion
0,Alimentation,1.511234e+07
1,Cosmétiques,3.618923e+06
2,Agroalimentaire,1.119037e+07
3,Agriculture,2.146560e+06
4,Pharmacie,4.481771e+06


In [ ]:
total_sum = new_df_applications["Proportion"].sum()
new_df_applications["Proportion"] = round(new_df_applications["Proportion"] / total_sum, 2)
new_df_applications

,Application,Proportion
0,Alimentation,0.41
1,Cosmétiques,0.10
2,Agroalimentaire,0.31
3,Agriculture,0.06
4,Pharmacie,0.12


In [ ]:
fig_2 = px.pie(new_df_applications, values="Proportion", names="Application", title="Utilisation des algues dans le monde", color='Application', color_discrete_map={'Alimentation':'#1D3B6E','Agroalimentaire':'#5B8FCB','Cosmétiques':'#209490','Agriculture':'#C73175', 'Pharmacie': '#F6A01E'})
fig_2

In [ ]:
fig_2.write_html("test_pie.html", include_plotlyjs="cdn")

## Top 20 espèces d'algues produites en Europe

In [ ]:
df_prod_alguae_europe = df_prod_alguae[df_prod_alguae["COUNTRY.UN_CODE"].isin(list_europe)].rename(columns={"COUNTRY.UN_CODE": "UN_Code", "PERIOD": "Année","VALUE" : "Production"}).merge(df_countries_filter, on="UN_Code", how="left").rename(columns={"Name_Fr": "Pays", "Continent_Group_Fr": "Continent"})
df_prod_alguae_europe_2022 = df_prod_alguae_europe[df_prod_alguae_europe["Année"] == 2022].rename(columns={"COUNTRY.UN_CODE": "UN_Code", "Year": "Année","VALUE" : "Production", "SPECIES.ALPHA_3_CODE": "3A_Code", "PRODUCTION_SOURCE_DET.CODE": "source_production"})
df_prod_alguae_europe_2022_by_species_top20 = df_prod_alguae_europe_2022[["Production", "3A_Code"]].groupby("3A_Code").sum().sort_values(by="Production", ascending=False).head(20).merge(df_species_filter_alguae[["3A_Code", "Scientific_Name"]], on="3A_Code", how="left").rename(columns={"Scientific_Name": "Espèce"})
df_prod_alguae_europe_2022_by_species_top20

,3A_Code,Production,Espèce
0,LAH,173198.838,Laminaria hyperborea
1,ASN,60025.232,Ascophyllum nodosum
2,LQD,44568.661,Laminaria digitata
3,SWB,36177.516,Phaeophyceae
4,SWX,2508.932,Algae
5,SWR,1623.848,Rhodophyta
6,SWG,805.010,Chlorophyceae
7,UDP,506.647,Undaria pinnatifida
8,GEL,416.590,Gelidium spp
9,FKU,380.500,Furcellaria lumbricalis


In [ ]:
fig = px.histogram(df_prod_alguae_europe_2022_by_species_top20, text_auto=True, x="Espèce", y="Production", color="Espèce", title="Repartition de la Production 2022 (en tonnes) en Europe selon l'espèce d'algues")
fig.show()

Les 20 premières algues les plus produites (en tonnes) en Europe sont les suivantes (par ordre décroissant) :
- Laminaria hyperborea (macro => production alginate)
- Ascophyllum nodosum (macro => biostimulant/fertilisants)
- Laminaria digitata (macro => production alginate)
- Phaeophyceae (macro)
- Algae (macro ?)
- Rhodophyta (macro)
- Chlorophyceae (micro)
- Undaria pinnatifida (macro = wakame)
- Gelidium spp (macro)
- Furcellaria lumbricalis (macro)
- Gelidium corneum (macro => l'agar-agar)
- Spirulina spp (spiruline)
- Saccharina latissima (macro = varech)
- Alaria esculenta (macro = dabberlocks)
- Himanthalia elongata (macro)
- Porphyra linearis (macro)
- Palmaria palmata (macro = dulse)
- Plantae aquaticae (macro ?)
- Vertebrata lanosa (macro)
- Porphyra spp (macro)

## Production totale d'algues en Europe

In [ ]:
prod_alguae_europe_2022 = df_prod_alguae_europe_2022["Production"].sum()
prod_alguae_europe_2022

np.float64(321244.85099999997)

**La production totale d'algues tout type en 2022 en Europe est de l'ordre de 320 000 tonnes**

## Répartition de la production 2022 d'algues en France selon l'espèce et le type de production

In [ ]:
df_prod_alguae_europe_2022_by_countrY_species = df_prod_alguae_europe_2022[["Pays", "3A_Code", "Production", "source_production"]].sort_values(by=["Pays", "Production"], ascending=False).merge(df_species_filter_alguae[["3A_Code", "Scientific_Name"]], on="3A_Code", how="left").rename(columns={"Scientific_Name": "Espèce"})
df_prod_alguae_europe_2022_by_countrY_species["source_production"] = df_prod_alguae_europe_2022_by_countrY_species["source_production"].apply(lambda x: "Récolte" if x == "CAPTURE" else "Aquaculture")
df_prod_alguae_europe_2022_by_countrY_species

,Pays,3A_Code,Production,source_production,Espèce
0,Îles Féroé,LQX,85.000,Aquaculture,Saccharina latissima
1,Îles Féroé,AJC,20.000,Aquaculture,Alaria esculenta
2,Îles Féroé,LQD,10.000,Aquaculture,Laminaria digitata
3,Portugal,SWR,1206.900,Récolte,Rhodophyta
4,Portugal,SWR,10.000,Aquaculture,Rhodophyta
...,...,...,...,...,...
56,Espagne,SWX,0.003,Récolte,Algae
57,Espagne,JNR,0.002,Récolte,Jania rubens
58,Espagne,GKA,0.002,Récolte,Gracilaria dura
59,Danemark,LQX,7.500,Aquaculture,Saccharina latissima


In [ ]:
df_prod_alguae_europe_2022_by_species_france = df_prod_alguae_europe_2022_by_countrY_species[df_prod_alguae_europe_2022_by_countrY_species["Pays"] == "France"]
# df_prod_alguae_europe_2022_by_species_france["source_production"] = df_prod_alguae_europe_2022_by_species_france["source_production"].apply(lambda x: "Récolte" if x == "CAPTURE" else "Aquaculture")
df_prod_alguae_europe_2022_by_species_france
fig = px.histogram(df_prod_alguae_europe_2022_by_species_france, text_auto=True, x="Espèce", y="Production", color="source_production", title="Repartition de la Production 2022 (en tonnes) en France selon l'espèce d'algues et le type de production")
fig.show()

In [ ]:
from plotly.subplots import make_subplots
import plotly.graph_objects as go
df_france_aqua = df_prod_alguae_europe_2022_by_species_france[df_prod_alguae_europe_2022_by_species_france["source_production"] == "Aquaculture"]
df_france_recolte = df_prod_alguae_europe_2022_by_species_france[df_prod_alguae_europe_2022_by_species_france["source_production"] == "Récolte"]
fig = make_subplots(rows=1, cols=2)

fig.add_trace(go.Bar(
    x=df_france_aqua["Espèce"],
    y=df_france_aqua["Production"],
), row=1, col=1)

fig.add_trace(go.Bar(
    x=df_france_recolte["Espèce"],
    y=df_france_recolte["Production"],
), row=1, col=2)
# fig = px.histogram(df_prod_alguae_europe_2022_by_species_france[df_prod_alguae_europe_2022_by_species_france["source_production"] == "Aquaculture"], text_auto=True, x="Espèce", y="Production", color="Espèce", title="Repartition de la Production (en tonnes) en France selon l'espèce d'algues en aquaculture")
fig.update_layout(
    showlegend=False,
    title="Repartition de la Production 2022 (en tonnes) en France selon l'espèce d'algues en aquaculture (gauche) et en récolte (droite)",
    yaxis_title_text='Production', # yaxis label
    # bargap=0.7, # gap between bars of adjacent location coordinates

)

fig.show()

## Répartition de la production 2022 d'algues en Norvège (1er producteur en volume en Europe) selon l'espèce et le type de production

In [ ]:
df_prod_alguae_europe_2022_by_species_norway = df_prod_alguae_europe_2022_by_countrY_species[df_prod_alguae_europe_2022_by_countrY_species["Pays"] == "Norvège"]
# df_prod_alguae_europe_2022_by_species_norway["source_production"] = df_prod_alguae_europe_2022_by_species_norway["source_production"].apply(lambda x: "Récolte" if x == "CAPTURE" else "Aquaculture")
fig = px.histogram(df_prod_alguae_europe_2022_by_species_norway, text_auto=True, x="Espèce", y="Production", color="source_production", title="Repartition de la Production 2022 (en tonnes) en Norvège selon l'espèce d'algues et le type de production")
fig.show()

In [ ]:
df_norway_aqua = df_prod_alguae_europe_2022_by_species_norway[df_prod_alguae_europe_2022_by_species_norway["source_production"] == "Aquaculture"]
df_norway_recolte = df_prod_alguae_europe_2022_by_species_norway[df_prod_alguae_europe_2022_by_species_norway["source_production"] == "Récolte"]
fig = make_subplots(rows=1, cols=2)

fig.add_trace(go.Bar(
    x=df_norway_aqua["Espèce"],
    y=df_norway_aqua["Production"],
), row=1, col=1)

fig.add_trace(go.Bar(
    x=df_norway_recolte["Espèce"],
    y=df_norway_recolte["Production"],
), row=1, col=2)
# fig = px.histogram(df_prod_alguae_europe_2022_by_species_norway[df_prod_alguae_europe_2022_by_species_norway["source_production"] == "Aquaculture"], text_auto=True, x="Espèce", y="Production", color="Espèce", title="Repartition de la Production (en tonnes) en norway selon l'espèce d'algues en aquaculture")
fig.update_layout(
    showlegend=False,
    title="Repartition de la Production 2022 (en tonnes) en Norvège selon l'espèce d'algues en aquaculture (gauche) et en récolte (droite)",
    yaxis_title_text='Production', # yaxis label
    # bargap=0.7, # gap between bars of adjacent location coordinates

)

fig.show()

## Répartition de la production 2022 d'algues en Europe selon l'espèce et selon le pays

In [ ]:
fig = px.histogram(df_prod_alguae_europe_2022_by_countrY_species, x="Pays", y="Production", color="Espèce", title="Repartition de la Production (en tonnes) selon l'espèce d'algues par Pays d'Europe")
fig.show()

## Critiques de  Dominykas Šalvaitis (PhD Seaweed in Norway, contact de Malaury)

First of all, **the first commercial license to culitvate seaweed in Norway was issued only in 2014, so industry itself is just a decade old**.

It is important to separate wild harvesting and cultivation. The main two species that are cultivated are Saccharina latissima and Alaria esculenta.**The main reason why these 2 species have been cultivated the most is that Saccharina latissima has been produced in Asia for a very long time, so Saccharina latissima became sort of a model species to work**. The lifecycle was known, the seedling production wasnt too complicates and it grows fast. Same goes for Alaria esculenta in terms of life cycle and ease of production.

When it comes to wild harvesting, the two species that are harvested have very specific purposes. **Laminaria hyperborea is used for alginate production and it has a great quality alginate, compared to Saccharina latissima, which has a lower quality.
While Aschophyllum nodosum is used for production of biostimulants, kind of a fertilizer.**
So, I dont actually see changes in cultivation or harvesting, onny in ammounts being produced. Which is rising, but very very slowly.

If people actually ate it, there would be more produced. **There are a few concerns of consuming Saccharina latissima due to high contents of iodine. It ranges from 2000mg/kg to 6000mg/kg and beyond. Plus, there isn't really a market for it in Scandinavia, thus adoption of Sugar Kelp in diets is difficult**.

## Analyse socio-éco des entreprises d'algues en Europe

In [79]:
df_ca_alguae = pd.read_csv('/content/sample_data/Algae-industry-Europe-socioeconomic - Socio-economic data.csv')
df_ca_alguae.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 548 entries, 0 to 547
Data columns (total 11 columns):
 #   Column                                          Non-Null Count  Dtype  
---  ------                                          --------------  -----  
 0   ID                                              548 non-null    object 
 1   Country                                         548 non-null    object 
 2   Organism group                                  548 non-null    object 
 3   Step in value chain                             548 non-null    object 
 4   Average available turnover in the last 5 years  253 non-null    object 
 5   Number of employees                             312 non-null    object 
 6   % of business focused on algae - ESTIMATED      380 non-null    object 
 7   % of business focused on algae - REFERENCE      317 non-null    object 
 8   Is Algae considered the main business stream    380 non-null    object 
 9   Unnamed: 9                                 

In [80]:
df_ca_alguae.rename(columns={"Organism group": "production_type", "Average available turnover in the last 5 years": "turnover_l5y"}).head(10)

,ID,Country,production_type,Step in value chain,turnover_l5y,Number of employees,% of business focused on algae - ESTIMATED,% of business focused on algae - REFERENCE,Is Algae considered the main business stream,Unnamed: 9,Unnamed: 10
0,AT01,Austria,Microalgae,Producing & processing,"16,061",15,100%,Website,Yes,NaN,NaN
1,AT02,Austria,Microalgae & Spirulina,Producing & processing,NaN,15,100%,Website,Yes,NaN,NaN
2,AT03,Austria,Spirulina,Producing & processing & services,NaN,5,100%,Website,Yes,NaN,NaN
3,AT04,Austria,Microalgae,Services,"6,350,815,930","27,232",2%,Website,No,NaN,NaN
4,AT05,Austria,Macroalgae,Processing,"6,754,249",37,33%,Website,No,NaN,NaN
5,AT06,Austria,Microalgae,Services,"88,460,310",292,2%,Website,No,NaN,NaN
6,BE01,Belgium,Microalgae,Producing & processing & services,NaN,NaN,NaN,NaN,NaN,NaN,NaN
7,BE02,Belgium,Microalgae,Producing,"552,097",5,100%,Website,Yes,NaN,NaN
8,BE03,Belgium,Spirulina,Producing & processing,NaN,NaN,NaN,NaN,NaN,NaN,NaN
9,BE04,Belgium,Algae sensu lato,Processing & services,"16,486,171",101,5%,Website,No,NaN,NaN


In [81]:
df_ca_alguae_bis = df_ca_alguae.rename(columns={"Organism group": "production_type", "Average available turnover in the last 5 years": "turnover_l5y", "Is Algae considered the main business stream": "is_algae_business", "% of business focused on algae - ESTIMATED": "algae_business_percent_estimated"})

In [82]:
df_ca_alguae_macro = df_ca_alguae_bis[df_ca_alguae_bis["production_type"] == "Macroalgae"]
df_ca_alguae_macro[(df_ca_alguae_macro["Country"]== "France")&(df_ca_alguae_macro["is_algae_business"] == "Yes")]



,ID,Country,production_type,Step in value chain,turnover_l5y,Number of employees,algae_business_percent_estimated,% of business focused on algae - REFERENCE,is_algae_business,Unnamed: 9,Unnamed: 10
175,FR01,France,Macroalgae,Producing & processing,"7,897,620",50,100%,Website,Yes,NaN,NaN
176,FR02,France,Macroalgae,Producing & processing,"1,426,535",18,50%,Internet,Yes,NaN,NaN
180,FR06,France,Macroalgae,Producing & processing,NaN,5,100%,Internet,Yes,NaN,NaN
182,FR08,France,Macroalgae,Producing & processing,"578,700",NaN,100%,Website,Yes,NaN,NaN
228,FR14,France,Macroalgae,Producing & processing & services,NaN,5,100%,Website,Yes,NaN,NaN
239,FR15,France,Macroalgae,Producing & processing,"6,458,500",NaN,100%,Website,Yes,NaN,NaN
250,FR16,France,Macroalgae,Producing & processing & services,NaN,40,66%,Website,Yes,NaN,NaN
261,FR17,France,Macroalgae,Producing & processing & services,"11,097,166",84,90%,Website,Yes,NaN,NaN
263,FR171,France,Macroalgae,Processing,"195,215",2,100%,Website,Yes,NaN,NaN
269,FR177,France,Macroalgae,Services,"115,000",5,100%,Website,Yes,NaN,NaN


In [83]:
df_ca_alguae_macro_clean = df_ca_alguae_bis[["ID", "Country", "production_type", "turnover_l5y", "is_algae_business"]]
df_ca_alguae_macro_clean.head(10)

,ID,Country,production_type,turnover_l5y,is_algae_business
0,AT01,Austria,Microalgae,"16,061",Yes
1,AT02,Austria,Microalgae & Spirulina,NaN,Yes
2,AT03,Austria,Spirulina,NaN,Yes
3,AT04,Austria,Microalgae,"6,350,815,930",No
4,AT05,Austria,Macroalgae,"6,754,249",No
5,AT06,Austria,Microalgae,"88,460,310",No
6,BE01,Belgium,Microalgae,NaN,NaN
7,BE02,Belgium,Microalgae,"552,097",Yes
8,BE03,Belgium,Spirulina,NaN,NaN
9,BE04,Belgium,Algae sensu lato,"16,486,171",No


In [85]:
df_ca_alguae_macro_clean_by_country = df_ca_alguae_macro_clean[(df_ca_alguae_macro_clean['is_algae_business'] == "Yes")&(df_ca_alguae_macro_clean['production_type'] == "Macroalgae")]
df_ca_alguae_macro_clean_by_country["turnover_l5y"] = df_ca_alguae_macro_clean_by_country["turnover_l5y"].str.replace(',', "").fillna(0)
df_ca_alguae_macro_clean_by_country["turnover_l5y"] = df_ca_alguae_macro_clean_by_country["turnover_l5y"].astype('int64')
df_ca_alguae_macro_clean_by_country = df_ca_alguae_macro_clean_by_country[["Country", "turnover_l5y"]].rename(columns={"turnover_l5y": "Chiffre_d_affaires", "Country": "Pays"}).groupby(by="Pays").sum().reset_index().sort_values(by="Chiffre_d_affaires", ascending=False)
df_ca_alguae_macro_clean_by_country

/tmp/ipykernel_16607/2087684375.py:2: SettingWithCopyWarning:


A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy

/tmp/ipykernel_16607/2087684375.py:3: SettingWithCopyWarning:


A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy



,Pays,Chiffre_d_affaires
2,France,54826349
4,Ireland,40403176
5,Norway,16196891
3,Iceland,8278434
7,Spain,7030840
6,Portugal,1045143
1,Estonia,855025
11,UK,800383
0,Denmark,23984
8,Sweden,23931


In [86]:
df_ca_alguae_macro_by_country = df_ca_alguae_macro_clean_by_country[df_ca_alguae_macro_clean_by_country["Chiffre_d_affaires"] > 0]

 '#0074E4' : FR
 '#1D3B6E' : Chine
 '#209490' : Indonésie
 '#C73175' : Norvège
 '#5B8FCB' : Corée du Sud
 '#C3ADD4' : Philippines
 '#F6A01E' : Autres pays d'Asie
 '#1E52A1' : Autres pays hors Asie
 '#13716A' : Chili
 '#991358' : Inde
 '#E5800B' : Japon
 '#CDE8FA' : Etats-Unis
 '#3573B9' : Autres pays (du monde)
 '#58B7B0' : Irlande
 '#D85F9F' : Islande
 '#F9C06B' : Russie
 '#E1D3E9' : Royaume-Uni
 '#F18882' : Autres pays d'Europe

In [87]:
def translate_country(row):
  if (row["Pays"] == "Ireland"):
    return 'Irlande'
  elif (row["Pays"] == "Norway"):
    return 'Norvège'
  elif (row["Pays"] == "Iceland"):
    return 'Islande'
  elif (row["Pays"] == "Spain"):
    return 'Espagne'
  elif (row["Pays"] == "Estonia"):
    return 'Estonie'
  elif (row["Pays"] == "UK"):
    return 'Royaume-Uni'
  elif (row["Pays"] == "Denmark"):
    return 'Danemark'
  elif (row["Pays"] == "Sweden"):
    return 'Suède'
  else:
    return row["Pays"]

In [91]:
colors= ['#0074E4','#58B7B0', '#C73175', '#D85F9F', '#991358','#F6A01E', '#1E52A1', '#E1D3E9', '#13716A','#E5800B', '#3573B9', '#CDE8FA', '#58B7B0', '#D85F9F', '#F9C06B', '#97C7EC', '#E1D3E9', '#F18882', '#FBDDD9', '#C3ADD4', '#F8C2BB']
df_ca_alguae_macro_by_country["Pays"] = df_ca_alguae_macro_by_country.apply(translate_country, axis=1)
fig_10 = go.Figure(
    data=[
        go.Bar(x=df_ca_alguae_macro_by_country["Pays"],
               y=df_ca_alguae_macro_by_country["Chiffre_d_affaires"],
               marker_color=colors,
               hovertemplate="""<span style='font-family:Parkinsans; font-weight:600'>Pays : %{x}</span><br>""" + \
               """<span style='font-family:Parkinsans; font-weight:600'>Chiffre d'affaires : %{y}</span>""" + \
      """<extra></extra>""",
        )
    ],
    layout=dict(
        barcornerradius=15,
    ),
)
fig_10.update_layout(
    title_text="Chiffre d’affaires réalisé (en euros)<br>par pays en Europe grâce à la<br>production d’algues",
    font_family="Parkinsans",
    title_font_weight=800,
    font_weight=600,
    font_color="#1D3B6E",
    paper_bgcolor = 'white',  # Fully transparent background
    plot_bgcolor = 'white',   # Fully transparent plot area

  )
fig_10.show()

/tmp/ipykernel_16607/1719756322.py:2: SettingWithCopyWarning:


A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy



In [92]:
fig_10.write_html("chiffre_affaires_europe_par_pays_mobile.html", include_plotlyjs="cdn")